# 무역 서류 패키지 생성 엔진 (v5.1 - 법률 리스크 테스트 특화)

이 노트북은 데이터를 생성하는 단계와 PDF를 렌더링하는 단계를 분리하여 제어할 수 있습니다.
모든 결과물은 `data/Documents/test_data_scenario/` 폴더에 시나리오 ID별로 저장됩니다.

## 1. 환경 설정

In [2]:
import base64
import os
import json
import random
import asyncio
from pathlib import Path
from google import genai
from dotenv import load_dotenv
from jinja2 import Template
from playwright.async_api import async_playwright

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODEL_ID = 'gemini-3.1-flash-lite-preview'

# 출력 경로 고정
BASE_OUTPUT_DIR = Path("data/Documents/test_data_scenario")
TEMPLATE_DIR = Path("data/Documents/form")
ASSET_DIR = TEMPLATE_DIR / "assets"
PROMPT_PATH = BASE_OUTPUT_DIR / "prompt.txt"
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[System] 작업 경로: {BASE_OUTPUT_DIR}")

[System] 작업 경로: data/Documents/test_data_scenario


## 2. [Step A] 데이터 생성 (Gemini 호출)

지정된 시나리오 ID 폴더에 `master.json`을 생성합니다.
**README.md에 정의된 리스크 시나리오에 맞춰 프롬프트를 조정하여 실행하세요.**

In [ ]:
SCENARIO_ID = "4" # 시나리오 번호 (4~20)
ADDITIONAL_RISK_INSTRUCTION = """
[Scenario 04 Specific Instruction]
1. Incoterms Risk: Use FOB for Air Freight (Inappropriate term usage).
2. Logistics Risk: Include a clause where the Seller charges warehouse fees to the Buyer under FOB terms (Cost allocation conflict).
"""

TARGET_DIR = BASE_OUTPUT_DIR / SCENARIO_ID
TARGET_DIR.mkdir(parents=True, exist_ok=True)

async def generate_scenario_data():
    with open(PROMPT_PATH, "r", encoding="utf-8") as f: base_prompt = f.read()
    
    full_prompt = base_prompt + "\n\n" + ADDITIONAL_RISK_INSTRUCTION
    
    print(f"[System] 시나리오 {SCENARIO_ID} 데이터 생성 중 (Gemini 호출)...")
    res = client.models.generate_content(
        model=MODEL_ID, contents=[full_prompt], config={'response_mime_type': 'application/json'}
    )
    # 최강의 JSON 파싱 로직 (뒤에 붙는 군더더기 텍스트 완전 무시)
    raw_res = res.text.strip()
    start_idx = raw_res.find('{')
    if start_idx != -1:
        raw_res = raw_res[start_idx:]
    try:
        data, _ = json.JSONDecoder().raw_decode(raw_res)
    except Exception as e:
        print(f"   ❌ JSON 파싱 실패: {e}")
        data = {}
    
    with open(TARGET_DIR / "master.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

        # [추가] 전략적 불일치 요소를 별도 파일로 추출하여 저장
        discrepancies = []
        docs = data.get('documents', {})
        for doc_name, doc_content in docs.items():
            if doc_content.get('is_signed') == False:
                discrepancies.append({'doc': doc_name, 'issue': 'Missing Signature'})
            if doc_content.get('has_shipped_stamp') == False:
                discrepancies.append({'doc': doc_name, 'issue': 'Missing Logistics Stamp'})
            if doc_content.get('discrepancy_note'):
                discrepancies.append({'doc': doc_name, 'issue': 'AI Noted Discrepancy', 'detail': doc_content['discrepancy_note']})
        
        if discrepancies:
            with open(TARGET_DIR / "discrepancies.json", "w", encoding="utf-8") as f:
                json.dump(discrepancies, f, indent=4, ensure_ascii=False)
            print(f"⚠️ 전략적 불일치 발견: {TARGET_DIR / 'discrepancies.json'}")
    
    print(f"✅ master.json 생성 완료: {TARGET_DIR / 'master.json'}")
    return data

trade_data = await generate_scenario_data()

[System] 시나리오 4 데이터 생성 중 (Gemini 호출)...
✅ master.json 생성 완료: data/Documents/test_data_scenario/4/master.json


## 3. [Step B] PDF 렌더링 (파일 기반)

저장된 `master.json`을 읽어 실제 PDF 서류를 생성합니다.
데이터 수정 후 이 셀만 다시 실행하여 PDF를 갱신할 수 있습니다.

In [7]:
RENDER_ID = "20" # PDF를 렌더링할 시나리오 번호
SOURCE_DIR = BASE_OUTPUT_DIR / RENDER_ID

async def render_pdfs():
    # [추가] 에셋을 Base64로 미리 로드하여 이미지 깨짐 방지
    assets_b64 = {}
    for img_path in Path('/mnt/c/Users/ysj58/github/Final_Project/data/Documents/form/assets').glob('*.png'):
        with open(img_path, 'rb') as f:
            assets_b64[img_path.name] = f"data:image/png;base64,{base64.b64encode(f.read()).decode()}"
    json_path = SOURCE_DIR / "master.json"
    if not json_path.exists():
        print(f"❌ 파일을 찾을 수 없습니다: {json_path}")
        return
        
    with open(json_path, "r", encoding="utf-8") as f: all_data = json.load(f)
        
    master = all_data.get('master_data') or {}
    docs = all_data.get('documents') or {}
    
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        context = await browser.new_context(viewport={'width': 794, 'height': 1123}, device_scale_factor=2)
        page = await context.new_page()
        
        # 문서별 랜덤 템플릿 선택 함수
        def pick_tpl(pattern):
            tpls = list(TEMPLATE_DIR.glob(pattern))
            if not tpls: return None
            chosen = random.choice(tpls)
            print(f"   🎲 Template Picked: {chosen.name} for {pattern}")
            return chosen
            
        render_configs = [
            ("sales_contract.pdf", "sales_agreement*.html", 'contract'),
            ("letter_of_credit.pdf", "lc*.html", 'lc'),
            ("bill_of_lading.pdf", "bl*.html", 'bl'),
            ("air_waybill.pdf", "awb*.html", 'awb'),
            ("commercial_invoice.pdf", "invoice*.html", 'invoice'),
            ("packing_list.pdf", "pl*.html", 'pl')
        ]
        
        for output_fn, tpl_pattern, data_key in render_configs:
            if data_key not in docs: continue
            
            # BL/AWB 특수 처리 (데이터에 awb가 있으면 awb 템플릿 우선)
            if data_key == 'bl' and 'awb' in docs: continue 
            
            tpl_path = pick_tpl(tpl_pattern)
            if not tpl_path or not tpl_path.exists(): continue
            
            with open(tpl_path, "r", encoding="utf-8") as f: html_template = f.read()
            
            # 렌더링 컨텍스트 (정합성 지원 고도화)
            doc_data = docs.get(data_key)
            if not isinstance(doc_data, dict):
                print(f"   ⚠️ Skip: '{data_key}' data is missing or invalid (None).")
                continue
            logistics = master.get('logistics') or {}
            
            # 항구 명칭 추출 (객체/문자열 대응)
            def get_port_name(port_data):
                if isinstance(port_data, dict): return port_data.get('name', '')
                return str(port_data)

            render_context = {
                **doc_data,
                "data_key": data_key,
                "data": doc_data,
                "master": master,
                "full_data": all_data,
                "documents": docs,
                "parties": master.get('parties') or {},
                "seller": master.get('parties', {}).get('exporter') or {},
                "buyer": master.get('parties', {}).get('importer') or {},
                "shipper": doc_data.get('shipper') or master.get('parties', {}).get('exporter') or {},
                "consignee": doc_data.get('consignee') or master.get('parties', {}).get('importer') or {},
                "notify_party": doc_data.get('notify_party') or master.get('parties', {}).get('notify_party') or "SAME AS CONSIGNEE",
                
                # 물류 정보 플래닝 (템플릿 편의성)
                "port_loading": get_port_name(logistics.get('port_loading')),
                "port_discharge": get_port_name(logistics.get('port_discharge')),
                "delivery_terms": doc_data.get('incoterms') or logistics.get('incoterms', ''),
                "vessel": logistics.get('vessel_name', ''),
                "voyage_no": logistics.get('voyage_no', ''),
                "shipment_deadline": doc_data.get('shipment_deadline') or master.get('dates', {}).get('latest_shipment_date', ''),
                "contract_date": doc_data.get('date') or master.get('dates', {}).get('contract_date', ''),
                "contract_no": doc_data.get('contract_no') or docs.get('contract', {}).get('contract_no', ''),
                # 리얼리티 증강 변수 (랜덤 요소)
                "assets": assets_b64,
                "stamp_rotation": random.randint(-15, 15),
                "stain_opacity": random.uniform(0.1, 0.4) if random.random() > 0.5 else 0,
                "stain_pos_x": random.randint(0, 500),
                "stain_pos_y": random.randint(0, 800),
                "has_texture": random.choice([True, False]),
                "has_shipped_stamp": doc_data.get('has_shipped_stamp', True),
                # Backward Compatibility Polyfills
                "transport_details": f"{logistics.get('vessel_name', '')} {logistics.get('voyage_no', '')}",
                "payment_info": doc_data.get('payment_terms') or master.get('terms', {}).get('payment_method', ''),
                "marks": doc_data.get('marks_and_nos') or master.get('product', {}).get('marks_numbers', ''),
                "description": doc_data.get('description_of_goods') or master.get('product', {}).get('name', ''),
            }
            
            try:
                html_content = Template(html_template).render(**render_context)
                await page.set_content(html_content)
                await page.pdf(path=str(SOURCE_DIR / output_fn), format="A4", print_background=True)
                print(f"✅ PDF 생성 완료: {output_fn}")
            except Exception as e:
                print(f"❌ {output_fn} 렌더링 에러: {e}")
            
        await browser.close()
        print(f"\n[Success] 시나리오 {RENDER_ID} PDF 생성 완료.")

await render_pdfs()

   🎲 Template Picked: sales_agreement_v2.html for sales_agreement*.html
✅ PDF 생성 완료: sales_contract.pdf
   🎲 Template Picked: lc_v3.html for lc*.html
✅ PDF 생성 완료: letter_of_credit.pdf
   🎲 Template Picked: bl_v5.html for bl*.html
✅ PDF 생성 완료: bill_of_lading.pdf
   🎲 Template Picked: invoice_v2.html for invoice*.html
✅ PDF 생성 완료: commercial_invoice.pdf
   🎲 Template Picked: pl_v2.html for pl*.html
✅ PDF 생성 완료: packing_list.pdf

[Success] 시나리오 20 PDF 생성 완료.


## 4. [Automation] 배치 처리 자동화

이 섹션은 `generation_plan.csv` 파일을 기반으로 여러 시나리오를 한꺼번에 생성하고 렌더링합니다.
각 생성 사이에는 **20초의 대기 시간**이 자동으로 적용됩니다.

In [3]:
import pandas as pd
import time

PLAN_CSV_PATH = BASE_OUTPUT_DIR / "generation_plan.csv"

async def batch_generate_scenarios(start_id, end_id):
    if not PLAN_CSV_PATH.exists():
        print(f"❌ 계획 파일이 없습니다: {PLAN_CSV_PATH}")
        return

    plan_df = pd.read_csv(PLAN_CSV_PATH)
    # ID 범위 필터링
    target_plan = plan_df[(plan_df['id'] >= start_id) & (plan_df['id'] <= end_id)]
    
    with open(PROMPT_PATH, "r", encoding="utf-8") as f: 
        base_prompt = f.read()

    for _, row in target_plan.iterrows():
        sid = str(row['id'])
        target_dir = BASE_OUTPUT_DIR / sid
        target_dir.mkdir(parents=True, exist_ok=True)
        if (target_dir / "master.json").exists():
            print(f"⏩ Skip: 시나리오 {sid} 이미 master.json이 존재합니다.")
            continue
        
        # 리스크 지침 결합
        risk_instruction = f"\n\n[Scenario {sid} Specific Instruction]\n"
        risk_instruction += f"- Risk 1: {row['risk_1']} (Basis: {row['legal_basis_1']})\n"
        risk_instruction += f"- Risk 2: {row['risk_2']} (Basis: {row['legal_basis_2']})\n"
        
        full_prompt = base_prompt + risk_instruction
        
        print(f"\n[System] 시나리오 {sid} 데이터 생성 중 (Gemini 호출)... Checkpoint 20s wait logic applied.")
        try:
            res = client.models.generate_content(
                model=MODEL_ID, 
                contents=[full_prompt], 
                config={'response_mime_type': 'application/json'}
            )
            # 최강의 JSON 파싱 로직 (뒤에 붙는 군더더기 텍스트 완전 무시)
            raw_res = res.text.strip()
            start_idx = raw_res.find('{')
            if start_idx != -1:
                raw_res = raw_res[start_idx:]
            try:
                data, _ = json.JSONDecoder().raw_decode(raw_res)
            except Exception as e:
                print(f"   ❌ JSON 파싱 실패: {e}")
                data = {}
            
            with open(target_dir / "master.json", "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            
            # 전략적 불일치 추출
            discrepancies = []
            docs = data.get('documents', {})
            for doc_name, doc_content in docs.items():
                if doc_content.get('is_signed') == False:
                    discrepancies.append({'doc': doc_name, 'issue': 'Missing Signature'})
                if doc_content.get('has_shipped_stamp') == False:
                    discrepancies.append({'doc': doc_name, 'issue': 'Missing Logistics Stamp'})
                if doc_content.get('discrepancy_note'):
                    discrepancies.append({'doc': doc_name, 'issue': 'AI Noted Discrepancy', 'detail': doc_content['discrepancy_note']})
            
            if discrepancies:
                with open(target_dir / "discrepancies.json", "w", encoding="utf-8") as f:
                    json.dump(discrepancies, f, indent=4, ensure_ascii=False)
                print(f"   ⚠️ 전략적 불일치 기록됨.")
                
            print(f"✅ 시나리오 {sid} 생성 완료.")
            
        except Exception as e:
            print(f"❌ 시나리오 {sid} 생성 에러: {e}")
            
        if sid != str(target_plan.iloc[-1]['id']):
            print(f"⏳ 다음 생성을 위해 5초 대기 중...")
            time.sleep(5)

# 실행 예시: 21번부터 30번까지 생성
# await batch_generate_scenarios(21, 30)

In [4]:
async def batch_render_pdfs(start_id, end_id):
    # 에셋 로드 (한 번만 수행)
    assets_b64 = {}
    asset_path = Path('/mnt/c/Users/ysj58/github/Final_Project/data/Documents/form/assets')
    for img_path in asset_path.glob('*.png'):
        with open(img_path, 'rb') as f:
            assets_b64[img_path.name] = f"data:image/png;base64,{base64.b64encode(f.read()).decode()}"

    async with async_playwright() as p:
        browser = await p.chromium.launch()
        context = await browser.new_context(viewport={'width': 794, 'height': 1123}, device_scale_factor=2)
        
        for sid in range(start_id, end_id + 1):
            source_dir = BASE_OUTPUT_DIR / str(sid)
            json_path = source_dir / "master.json"
            
            if not json_path.exists():
                print(f"⏩ Skip: 시나리오 {sid} 폴더에 master.json이 없습니다.")
                continue
                
            print(f"\n[System] 시나리오 {sid} PDF 렌더링 시작...")
            with open(json_path, "r", encoding="utf-8") as f: 
                all_data = json.load(f)
            
            master = all_data.get('master_data', {})
            docs = all_data.get('documents', {})
            page = await context.new_page()
            
            render_configs = [
                ("sales_contract.pdf", "sales_agreement*.html", 'contract'),
                ("letter_of_credit.pdf", "lc*.html", 'lc'),
                ("bill_of_lading.pdf", "bl*.html", 'bl'),
                ("air_waybill.pdf", "awb*.html", 'awb'),
                ("commercial_invoice.pdf", "invoice*.html", 'invoice'),
                ("packing_list.pdf", "pl*.html", 'pl')
            ]
            
            for output_fn, tpl_pattern, data_key in render_configs:
                if data_key not in docs: continue
                if data_key == 'bl' and 'awb' in docs: continue 
                
                tpls = list(TEMPLATE_DIR.glob(tpl_pattern))
                if not tpls: continue
                tpl_path = random.choice(tpls)
                
                with open(tpl_path, "r", encoding="utf-8") as f: 
                    html_template = f.read()
                
                doc_data = docs.get(data_key)
                logistics = master.get('logistics', {})
                
                def get_port_name(port_data):
                    if isinstance(port_data, dict): return port_data.get('name', '')
                    return str(port_data)

                render_context = {
                    **doc_data, "data_key": data_key, "data": doc_data, "master": master,
                    "full_data": all_data, "documents": docs,
                    "parties": master.get('parties', {}),
                    "seller": master.get('parties', {}).get('exporter', {}),
                    "buyer": master.get('parties', {}).get('importer', {}),
                    "shipper": doc_data.get('shipper') or master.get('parties', {}).get('exporter', {}),
                    "consignee": doc_data.get('consignee') or master.get('parties', {}).get('importer', {}),
                    "notify_party": doc_data.get('notify_party') or master.get('parties', {}).get('notify_party') or "SAME AS CONSIGNEE",
                    "port_loading": get_port_name(logistics.get('port_loading')),
                    "port_discharge": get_port_name(logistics.get('port_discharge')),
                    "delivery_terms": doc_data.get('incoterms') or logistics.get('incoterms', ''),
                    "vessel": logistics.get('vessel_name', ''), "voyage_no": logistics.get('voyage_no', ''),
                    "shipment_deadline": doc_data.get('shipment_deadline') or master.get('dates', {}).get('latest_shipment_date', ''),
                    "contract_date": doc_data.get('date') or master.get('dates', {}).get('contract_date', ''),
                    "contract_no": doc_data.get('contract_no') or docs.get('contract', {}).get('contract_no', ''),
                    "assets": assets_b64, "stamp_rotation": random.randint(-15, 15),
                    "stain_opacity": random.uniform(0.1, 0.4) if random.random() > 0.5 else 0,
                    "stain_pos_x": random.randint(0, 500), "stain_pos_y": random.randint(0, 800),
                    "has_texture": random.choice([True, False]), "has_shipped_stamp": doc_data.get('has_shipped_stamp', True),
                    "transport_details": f"{logistics.get('vessel_name', '')} {logistics.get('voyage_no', '')}",
                    "payment_info": doc_data.get('payment_terms') or master.get('terms', {}).get('payment_method', ''),
                    "marks": doc_data.get('marks_and_nos') or master.get('product', {}).get('marks_numbers', ''),
                    "description": doc_data.get('description_of_goods') or master.get('product', {}).get('name', ''),
                }
                
                try:
                    html_content = Template(html_template).render(**render_context)
                    await page.set_content(html_content)
                    await page.pdf(path=str(source_dir / output_fn), format="A4", print_background=True)
                except Exception as e:
                    print(f"   ❌ {output_fn} 에러: {e}")
            
            await page.close()
            print(f"✅ 시나리오 {sid} PDF 생성 완료.")
        
        await browser.close()

# 실행 예시: 21번부터 30번까지 렌더링
# await batch_render_pdfs(21, 30)

In [6]:
# 실행 예시: 21번부터 30번까지 생성
await batch_generate_scenarios(21, 100)

⏩ Skip: 시나리오 21 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 22 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 23 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 24 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 25 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 26 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 27 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 28 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 29 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 30 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 31 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 32 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 33 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 34 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 35 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 36 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 37 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 38 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 39 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 40 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 41 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 42 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 43 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 44 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 45 이미 master.json이 존재합니다.
⏩ Skip: 시나리오 46 이미 master

In [7]:
# 실행 예시: 21번부터 30번까지 렌더링
await batch_render_pdfs(31, 100)


[System] 시나리오 31 PDF 렌더링 시작...
✅ 시나리오 31 PDF 생성 완료.

[System] 시나리오 32 PDF 렌더링 시작...
✅ 시나리오 32 PDF 생성 완료.

[System] 시나리오 33 PDF 렌더링 시작...
✅ 시나리오 33 PDF 생성 완료.

[System] 시나리오 34 PDF 렌더링 시작...
✅ 시나리오 34 PDF 생성 완료.

[System] 시나리오 35 PDF 렌더링 시작...
✅ 시나리오 35 PDF 생성 완료.

[System] 시나리오 36 PDF 렌더링 시작...
✅ 시나리오 36 PDF 생성 완료.

[System] 시나리오 37 PDF 렌더링 시작...
✅ 시나리오 37 PDF 생성 완료.

[System] 시나리오 38 PDF 렌더링 시작...
✅ 시나리오 38 PDF 생성 완료.

[System] 시나리오 39 PDF 렌더링 시작...
✅ 시나리오 39 PDF 생성 완료.

[System] 시나리오 40 PDF 렌더링 시작...
✅ 시나리오 40 PDF 생성 완료.

[System] 시나리오 41 PDF 렌더링 시작...
✅ 시나리오 41 PDF 생성 완료.

[System] 시나리오 42 PDF 렌더링 시작...
✅ 시나리오 42 PDF 생성 완료.

[System] 시나리오 43 PDF 렌더링 시작...
✅ 시나리오 43 PDF 생성 완료.

[System] 시나리오 44 PDF 렌더링 시작...
✅ 시나리오 44 PDF 생성 완료.

[System] 시나리오 45 PDF 렌더링 시작...
✅ 시나리오 45 PDF 생성 완료.

[System] 시나리오 46 PDF 렌더링 시작...
✅ 시나리오 46 PDF 생성 완료.

[System] 시나리오 47 PDF 렌더링 시작...
✅ 시나리오 47 PDF 생성 완료.

[System] 시나리오 48 PDF 렌더링 시작...
✅ 시나리오 48 PDF 생성 완료.

[System] 시나리오 49 PDF 렌더링 시작...
✅ 시나리오 49 PDF 